____
### 1. Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal


____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes:
1. observation space, 5 numbers as an array [Leftover Stock, Days Left, Last known demand]
2. action space (price of good, number over a continuouse range from 5 to 50)
3. max steps (days of simulation)
4. cost (cost incurred to obtain 1 unit)


In [2]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        reward = reward / 100.0
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        if truncated and self.inventory > 0:
            reward -= self.inventory * 2.0  # $2 penalty per unsold unit
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory / self.max_inventory, # 0 to 1
                        (self.max_steps - self.step_count) / self.max_steps, # 0 to 1
                        self.last_demand / self.max_inventory] # 0 to 1
                        , dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 0.8
        noise = np.random.normal(0, 2) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment

In [3]:
env = DynamicPricingEnv()

In [4]:
obs, info = env.reset()
print(f"Observation space representing: [stock left, days left, last known sold]: {obs}")
print(f"Extra info dictrionary: {info}")

obs, reward, terminated, truncated, info = env.step([10.0])
env.get_latest()
print(reward)
print(f"{terminated}, {truncated}")

Observation space representing: [stock left, days left, last known sold]: [1. 1. 0.]
Extra info dictrionary: {}
Leftover Stock: 0.6800000071525574 units, Days Left 0.9666666388511658, Sold Units: 0.3199999928474426
1.6
False, False


____
### 3. Deciding which model to use 
- A standard Q-table cannot be used as the action space (price of good) is a continuous number instead of a discrete number
- Therefore, there are several choices for neural networks
    1. Proximal Policy Optimization(PPO) Model
    2. Twin Delayed Deep Deterministic Policy Gradientt (TD3) Model
    3. Soft-Actor-Critic (SAC) Model

#### 3.1 Proximal Policy Optimization (PPO) Model
- a policy gradient method which directly learns "Given this state, what price should I output"
- the "Proximal" part of the PPO model adjusts the actions in small amounts to find the optimal policy 
- therefore, PPO Models limits how drastically the policy changes each update to prevent unstable training

##### PPO Architecture
- PPO uses 2 networks
1. Actor Network
    - Outputs the pricing policy
    - Given a state, feeds into a neutal network and outputs a pricing distribution
    - `state → neural network → price distribution`
    - The agent then samples prices around that range

2. Critic Network
    - Estimates the future rewards
    - Asks "How profitable is this situation" and helps the Actor Network improve

##### PPO Flow
`Observe market → Choose price → Simulate customer response → Get profit reward → Update pricing policy slightly`


#### 3.2 Twin Delayed Deep Deterministic Policy Gradient (TD3)
- Designed specefically for continuous action space, precise control and stable deep Q-learning
- instead learning "what action should I take?" the model learns "how good is a particular action"

##### TD3 Architecture
- TD3 uses 3 networks
1. Actor Network
    - Outputs a distribution over actions
    - `state → distribution`
    - Samples from the distribution to create exploration

2. Critic Network (2 critic networks)
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as PPO)

3. Replay Buffer
    - To store experiences and reuse them, making the TD3 highly sample efficient
    - Allow TD3 to learn from past experiences repeatedly

4. Target Networks

##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`
- Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)



#### 3.3 Soft Actor-Critic (SAC)
- Considered one of the strongest RL algorithms for continuous control
- Combines actor-critic learning, entropy maximisation and off-policy training
- Tries to maximise both `Reward` and `Exploration` instead of only `profit`
- Entropy refers to the epsilon (randomness of the actions) therefore it encourages the agent to keep exploring pricing options, preventing the model from becoming too deterministic 


##### SAC Architecture
- SAC uses 3 networks
1. Actor Network
    - Outputs the exact price
    - `state → price`

2. Critic Network (2 critic networks)
    - The "Twin" part is in refernce to the 2 critic networks
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` 

##### SAC Flow
- `Observe State → Sample action from policy distribution → receive reward → update critic → update actor → encourage exploration through entropy bonus`
- Reward is evaluated as `total reward = reward + entropy bonus` and to encourage exploration

____
### 4. Training the PPO Model
- In the spirit of learning, I will be training a PPO model first before training a TD3 model and a SAC model

#### 4.1 Establishing Architecture
- Actor and Critic networks are first created
    - Actor and Critic networks are used to observe the state and output the action and critic values, no learning logic implemented yet
- Create RolloutBuffer to store data and translate the data into learning signals 
    - Data is converted to learning signals, networks do not learn yet
- `ppo_update()` function converts teh learning signals and updates the networks via gradient descent
    - computes policy loss, value loss and entropy loss
- `train()` function combines the network, rollout buffer and `ppo_update()` to train the model 

#### 4.2 Creating Actor and Critic Networks

In [5]:
class ActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        # self.backbone acts as a shared extractor used by both the actor and critic network
        self.backbone = nn.Sequential( #nn.Sequential runs the layers in order, linear -> Tanh -> linear -> tanh
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(64, action_dim) 
        self.log_std = nn.Parameter(torch.ones(action_dim) * 2.0) 
        self.critic = nn.Linear(64, 1)

    def forward(self, obs): #needs to be overridden
        features = self.backbone(obs) #vector of 128 values
        mean = self.actor_mean(features) #obtain the action 
        std = self.log_std.exp().expand_as(mean) #obtain the std dev and fits the shape with with mean 
        critic_val = self.critic(features) #obtain the critic value
        return mean, std, critic_val
    
    def get_action(self, obs): #creating action based off the network
        mean, std, value = self.forward(obs) #calling forward to obtain values from actor and critic network
        dist = Normal(mean, std) #creates normal distribution
        action = dist.sample() #samples the distribution
        log_prob = dist.log_prob(action).sum(dim=-1) #obtains sum of log distribution (exp below)
        return action, log_prob, value.squeeze(-1) #converts the value into a scalar

    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1) #calculates the entropy of the normal distribution, how random or uncertain the distribution is 
        # entropy is summed up over the last dimension
        return log_prob, value.squeeze(-1), entropy

##### 4.21 Explanation of Code:
```python
self.backbone = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh()
        )
```
- `nn.sequential()` ensures that the following layers run in sequence
- `nn.Linear(obs_dim, 64)` -> y = Wx + b
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.Tanh()` -> y = tanh(x)
    - x is a `64 x 1 vector` that is the result from the linear vector
    - y is the resultant `64 x 1 vector` from applying the tanh() function on every single value in the original vector
    - this function introduces non-linearlity and squashes every value to be within the range (-1, 1)
    - the introduction of non-linearity allows the model to learn non-linear behaviours
- `nn.Linear(64, 64)` -> y = Wx + b
    - uses the non-linear outputs from the previous layers but turns them into more sophisticated outputs using weights and biases
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer

```python
self.actor_mean = nn.Linear(64, action_dim) 
self.log_std = nn.Parameter(torch.ones(action_dim) * 2.0)
self.critic = nn.Linear(64, 1)
```
- `self.actor_mean = nn.Linear(64, action_dim)`
    - Makes use of the 64 outputs from the backbone to derive the outputs in the action dimension (mean of the price)
- `self.log_std = nn.Parameter(torch.zeroes(action_dim) * 2.0)`
    - std deviation measures the randomness of the distributionm, log(std) is used so that the value is not -ve, later converted using `exp()`
    - wrapping it in `nn.Parameter()` means that the value should be learning during training
    - this makes it such that the PPO model learns during training what is the appropriate level of exploration
- `self.critic = nn.Linear()`
    - Makes use of the 64 outputs from the backbone to derive 1 value, which represents the future reward expected from the current state

- `log_prob = dist.log_prob(action).sum(dim=-1)`
    - `dist.log_prob(action)` obtains the log_probability of the action within the distribution
    - `.sum(dim=-1)` sums it along the last dimension
    - instad of multiplying individual probabilities, it adds up `log(prob)` instead 

##### 4.22 Over-Arching View:
- observation (vector of 3 values) -> self.backbone(vector of 64 values) 
- The observation in the form of 64 values is then fed into the actor network and critic network
- ie `self.backbone(vector of 64 values) -> action (1 value)` and `self.backbone(vector of 64 values) -> future reward expected (1 value)`

##### 4.3 Creating RolloutBuffer
- Acts as a container to store past experiences and convert them into learning signals
- `RolloutBuffer` converts raw experiences into 3 learning signals:
    1. Advantages: How much better an action was than expected
    2. Returns: The total expected future reward
    3. Normalized Advantages: Standardized advantages for stable training

In [6]:
class RolloutBuffer:
    def __init__(self):
        self.clear() #delegates initialisation to the clear() method

    def clear(self): #resets all lists to empty
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []

    def add(self, obs, action, log_prob, reward, value, done): #appends one round of experiences
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95): #compute generalized advantage est
        advantages = [] 
        gae = 0.0
        values = self.values + [last_value] # adds one extra value to compute next-step difference
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns

##### 4.31 Explanation of code

```python
def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95):
    advantages = [] 
    gae = 0.0
    values = self.values + [last_value] 
    for t in reversed(range(len(self.rewards))): 
        delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
        gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
        advantages.insert(0, gae)
    returns = [adv + val for adv, val in zip(advantages, self.values)]
    return advantages, returns
```
- `compute_returns()` is computes the Advantages ("How much better an action was than expected") and the returns ("total expected future rewards")
- Temporal Diff (TD) error: `delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]` measures the diff in outcome compared to the expected outcome (measuring suprise)
- Equation generalises to `TD = (actual outcome) - (expected outcome)`
    - delta < 0, worse than expected, delta > 0, better than expected
    - `self.rewards[t]` is the immediate reward from action 
    - `gamma` is the discount factor, measures how much the model cares about future rewards
    - `values[t + 1]` is the predicted future value, the critic's estimate of future reward after 1 step 
    - `gamma * values[t + 1]` is the expected future rewards, multiplying by `gamma` discounts the expected future rewards 
    (expected future rewards are slightly less vaulable)
    - `values[t]` is baseline prediction before seeing the outcome
    - `1 - self.dones[t]` acts as a switch, if the episode has not terminated at the step t, `self.dones[t] = 0`, if it has terminated, `self.dones[t] = 1` and `1 - self.dones[t] = 0`
    - When episode has ended, `1 - self.dones[t] = 0` and future term of `gamma * values[t + 1] * (1 - self.dones[t]) = 0`
    - Therefore, `delta = self.rewards[t] - values[t]` since there is no more `expected future rewards` and the difference in expectation is simply the rewards up to that point, `self.rewards[t]`, minus expected rewards `values[t]`

- Generalized Advantage Estimation: `gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae` which is a smoothened estimate of "how good was this action"
- Equation generalises to `gae = current TD error + discounted future gae`
    - starts out with the current step's TD error
    - adds the discounted future TD errors = `gamma * gae_lambda * (1 - self.dones[t]) * gae`
    - when the episode ends, `self.dones[t] = 1` and `1 - self.dones[t] = 0` and the future TD errors = 0 (no more future for the episode since it terminated) 

- `advantages.insert(0, gae)`
    - inserts the calculated gae at the **front** of the `advantages` array
    - when the loop runs in reverse during `for t in reversed(range(len(self.rewards))):` the first iteration takes the last timestep. with the calculated gae inserted at the front, the advantages are in forward order

- `returns = [adv + val for adv, val in zip(advantages, self.values)]` : equivalent to doing:
    ```python
    returns = []
    for adv, val in zip(advantages, self.values):
        r = adv + val
        returns.append(r)
    ```
    - for each step, returns = advantage + value estimate
    - adding it to the array saved as returns 

``` python
def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns
```
- `torch.tensor(data).to(device)`
    - Creates a new tensor from python data (list, arrays, numbers), `data -> tensor`
    - used for `obs`, `advantages` and `returns` as they are all raw lists of data
-  `torch.stack(tensors, dim = 0)`
    - joins multiple existing tensors along a new dimension
    - used for `actions` and `log_probs` as `self.actions` and `self.log_probs` is a list of tensors already
- `to(device)` is used to move a tensor to a specified device (CPU or GPU)
- `advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)`
    - `advantages - advantages.mean()` subtracts the mean from every value such that it is values in the tensors are now **centred around zero** 
    - `/(advantages.std() + 1e-8)` divides every value from the std deviation 
    - this process normalises all data within the tensor (tensor supports element-wise operations)


##### 4.32 Over-Arching View:
1. `add()` method
   - raw data is collected: observations, actions, log probabilities, rewards, values, and done flags
   - All stored as lists in the buffer

2. `compute_returns()` method
   - Compute TD errors (`delta`) by comparing actual outcomes to critic predictions
   - Accumulate GAE backward through time to get smooth advantage estimates
   - Combine advantages with value estimates to get target returns for the critic

3. `to_tensors()` method
   - Convert all lists to PyTorch tensors
   - Normalize advantages to have mean=0 and std=1 for stable training
   - Move tensors to the correct device (CPU or GPU)

4. **Learning Signal Output**
   - `advantages`: How much better/worse each action was compared to expected (used by actor)
   - `returns`: Target value estimates (used by critic)
   - Both aligned with original observations and actions for supervised learning

##### 4.4 Instantiate functions to update PPO

In [7]:
def ppo_update(model, optimizer, obs, actions, old_log_probs,advantages, returns, 
               clip_range=0.2, ent_coef=0.05, vf_coef=0.5, n_epochs=10, batch_size=64):
    total_steps = obs.shape[0] 
    for _ in range(n_epochs):
        indices = torch.randperm(total_steps)
        for start in range(0, total_steps, batch_size):
            idx = indices[start : start + batch_size]
            new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])
            ratio = (new_log_probs - old_log_probs[idx]).exp()
            adv = advantages[idx]
            policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
            value_loss = nn.functional.mse_loss(values, returns[idx])
            entropy_loss = -entropy.mean()
            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

##### 4.41 Explanation of code
- Required Parameters:
    1. `model` : ActorCritic neural network being trained
    2. `optimizer` : The optimizer that applies weight updates via backpropagation (adjusting weights of each matrix using backprop)
    3. `obs` : observations collected during rollout, `512x3 matrix`, converted to tensor (3 cols as observation space is 3 dimension)
    4. `actions` : actions taken during rollout, `512x1 vector`, converted to tensor (1 col as action space is just 1 value)
    5. `old_log_probs` : log prob of actions under the old policy (before update), used to compute importance sampling ratio
    6. `advantages` : normalized advantage estimates (array with 512 entries), tells us how good each action was compared to expected 
    7. `returns` : target returns for the critic (array with 512 entries), what the critic model should predict
- 512 -> max of 512 timesteps in an episode
- Hyperparameters:
    1. `clip_range=0.2` : clipping parameter, restricting the policy ration to [1-0.2, 1+0.2] = [0.8, 1.2], to prevent drastic policy changes
    2. `ent_coef=0.05` : weight on exploration bonus, provides incentive to explore 
    3. `vf_coef=0.5` : value function coefficient, the weight on critic loss - How important is predicting returns accurately
    4. `n_epochs=10` : number of passes through the data, extracting more learning signals each epock
    5. `batch_size=64` : batch size for gradient updates, process 64 timesteps at a time
    
- `total_steps = obs.shape[0]` 
    - `.shape` gives (rows, cols), therefore `obs.shape[0]` = rows of data = number of timesteps

```python
for _ in range(n_epochs):
    indices = torch.randperm(total_steps)
    for start in range(0, total_steps, batch_size):
```
- iterate through set number of times according to hyperparameters set
`indices = torch.randperm(total_steps)`
    - creates a tensor of the numbers from 0 to total_steps-1 in a **random order**
    - this shuffles the rollout data before training for each iteration so that the PPO does not see the data in the same order each time
    - this breaks any ordering bias and makes mini-batches random
`for start in range(0, total_steps, batch_size)`
    - iterates from 0 to total_steps, jumping by `batch_size` at once


- `idx = indices[start : start + batch_size]` : slices the array in chunks of length equal to `batch_size`, becomes the current batch fof sample indices
- `new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])` : evaluates the model based off the small batch selected
- `ratio = (new_log_probs - old_log_probs[idx]).exp()` : computes the PPO probability ratio, measuring how much the new policy changed compared to the old policy for those same actions (element wise operation for the whole array)
    - ratio = 1, policy is unchanged
    - ratio > 1, the new policy assigns higher probability to the action ()
    - ratio < 1, the new policy assigns lower probability to the action
- `adv = advantages[idx]` : obtains the advantage values for this batch of values
- ``` python 
    policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
    ```
- `ratio * adv` : element-wise operation between importance-sampling ratio and the advantage array for this batch, `adv`
- `torch.clamp(ratio, 1 - clip_range, 1 + clip_range)` : `torch.clamp(tensor, min, max)` clips all the values inside the ratio tensor to within the range of (1 - clip_range, 1 + clip_range)
    - if value < min or value > max , the value is set to min / max
    - `* adv` to perform element-wise operation between importance-sampling ratio and the advantage array for this batch
- `torch.min(tensor1, tensor2)` : compares the 2 tensors element-wise and extracts the lower value at each position, constructing a new tensor out of it
- `.mean()` : averages all values of the tensor to obtain the mean 
- calculating policy loss:
    - finding the **mean advantage value** after multiplying by ratio element-wise and then **limiting it** to within the range `(1-clip_range, 1+clip_range)`
    - multiplying by `-` then obtains the `policy loss` 

- `value_loss = nn.functional.mse_loss(values, returns[idx])` : the Mean Square Error between critic predictions and targets
    - `nn.functional.mse_loss(input_tensor, target_tensor)` finds the mean squared error between every value in the input and the target tensor
    - use `reduce = none` to get the output as a tensor, else it returns the average of all the MSE values
- `entropy_loss = -entropy.mean()` : multiplies the mean entropy by -1, turning the maximise entropy into a minimization objective
    - when entropy increases, loss decreases which is what gradient descent wants
- `loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss`
    - formula generalises to : overall loss = policy loss + overall value loss + overall entropy loss 
    - `overall value loss = vf_coef * value_loss`, where `vf_coef` scales the critic term in thte total loss equation (how strongly gradient descent prioritizes reducing the value, critic's accuracy, relative to policy and entropy)
    - `overall entropy loss = ent_coef * entropy_loss`, where `ent_coef` controls how strongly exploration is encouraged (how strongly gradient descent prioritizes exploration, relative to policy and value)
- `optimizer.zero_grad()` : clears the gradients stored inside the optimizer before calculations are done 
- `loss.backward()` : with loss represented as a function of policy loss, value loss and entropy loss, ie L = f(p,v,e)
    - calling `.backward()` differentiates loss wrt. p, v and e and stores it inside each parameter, which can be accessed with `.grad()`
- `nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)` : rescales `.grad()` in place if the total norm > max_norm, so that the parameter update magnitude is limited when `optimizer.step()` runs
- `optimizer.step()` : computes updates for each parameter's rule within the optimizer and updates it
    - this thus makes the new weight value of the neural network the model's current parameters    
    - this updates the whole network, meaning the backbone network, actor and critic network.

##### 4.42 Over-Arching View:
- The env produces 3‑D observations that a shared backbone encodes into features consumed by an `ActorCritic` network (actor outputs a Gaussian price policy; critic predicts state value).
- Episodes are collected into a `RolloutBuffer` (obs, actions, log-probs, rewards, values, dones); GAE computes advantages and returns from those rollouts.
- `ppo_update()` runs multiple epochs of minibatch updates with the clipped policy objective, value MSE, and entropy bonus to update the network.
- Repeat collect → compute (advantages/returns) → update until total timesteps, producing a trained pricing policy.

##### 4.5 Instantiate function to train model

In [8]:
def train(total_timesteps=200_000, n_steps=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    env = DynamicPricingEnv()
    obs_dim = env.observation_space.shape[0] # 3
    action_dim = env.action_space.shape[0] # 1
    model = ActorCritic(obs_dim, action_dim).to(device) # moves all of the model's tensors to the device
    optimizer = optim.Adam(model.parameters(), #register all params, backkbone, actor head, log_std, critic
                            lr=3e-4) # learning rate set at 3e-4
    buffer = RolloutBuffer()
    obs, _ = env.reset()
    episode_reward = 0
    episode_count = 0
    timestep = 0
    while timestep < total_timesteps: # runs until timestep budget is exhausted
        buffer.clear() #reset the buffer at the start of every rollout: old experience thrown away as they were collected under a previous version of the policy
        for _ in range(n_steps):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad(): #actions within this block temporarilty disables autograd and do not track history or build computation graphs
                action, log_prob, value = model.get_action(obs_tensor)
            action_np = action.cpu().numpy()[0]
            action_np = np.clip(action_np, 5.0, 50.0) 
            next_obs, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated
            buffer.add(obs = obs,
                       action = action.squeeze(0).cpu(),
                       log_prob = log_prob.squeeze(0).cpu(),
                       reward = reward,
                       value = value.squeeze(0).cpu().item(),
                       done = float(done))
            episode_reward += reward
            obs = next_obs
            timestep += 1
            if done:
                episode_count += 1
                if episode_count % 20 == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0
                obs, _  = env.reset()
        with torch.no_grad():
            last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            _, _, last_value = model.get_action(last_obs)
            last_value = last_value.squeeze(0).cpu().item()
        advantages, returns = buffer.compute_returns(last_value)
        obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
        ppo_update(model, optimizer, obs_t, act_t, lp_t, adv_t, ret_t)
    # torch.save(model.state_dict(), "ppo_pricing.pth")
    print("Training complete.")
    return model


##### 4.51 Explanation of code

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
env = DynamicPricingEnv()
obs_dim = env.observation_space.shape[0] # 3
action_dim = env.action_space.shape[0] # 1
model = ActorCritic(obs_dim, action_dim).to(device) 
optimizer = optim.Adam(model.parameters(), lr=3e-4) 
```
- `model = ActorCritic(obs_dim, action_dim).to(device)`
    - instantiate a model using the `obs_dim` and `action_dim` of the environment
    - using `.to(device)` moves the tensor of the model to `device`

- `optimizer = optim.Adam(model.parameters(), lr = 3e-4)`
    - creates a **Adaptive Moment Estimation** optimizer, updating the network's weights
    - runs 2 running statistics per parameter:
        1. The mean of gradients
        2. The mean of squared gradients 
    - `model.parameters()` as an input : registers all parameters of the model in the optimizer
    - `lr = 3e-4` : learning rate for the PPO, cannot be too low or too high

``` python
while timestep < total_timesteps:
    buffer.clear()
```
- outer loop runs until the total timestep budget is exhausted, `buffer.clear()` resets the buffer at the start of every rollout
- This ensures that old experiences are thrown away as they are collected under a previous version of the policy and no longer valid for the current update

```python
for _ in range(n_steps):
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
```
- `torch.tensor(obs, dtype=torch.float32)` : converts the NumPy array (obs) into a PyTorch Tensor
    - datatype set to `float.32` as that is the type used by neural networks
- `unsqueeze(0)` : inserts a dimension at postion 0
    - `nn.Linear` layers expect inputs shaped as `[batch_size, features]` while the `obs` tensor is shaped `[3]`
    - therefore, `.unsqueeze(0)` inserts a dimension at position 0, making it `[1, 3]`, a batch of size 1
    - this is required for matrix multiplication


```python
with torch.no_grad():
    action, log_prob, value = model.get_action(obs_tensor)
```
- `with torch.no_grad()`
    - disables gradient computation for the methods inside, which is `action, log_prob, value = model.get_action(obs_tensor)`
    - in context here, it disables computation of gradient for `model.get_action(obs_tensor)` because this portion is just meant for data collection
    - therefore, there is no need to update the model as it is simply used to make decisions 
    - this is more efficient for computation

```python
action_np = action.cpu().numpy()[0]
action_np = np.clip(action_np, 5.0, 50.0) 
next_obs, reward, terminated, truncated, _ = env.step(action_np)
done = terminated or truncated
```
- `action.cpu().numpy()[0]` : `.cpu().numpy()` moves the tensor to the cpu so that numy can access it and converts it to a numpy array
    - `[0]` is used to remove the batch dimension, converting it from an array shaped `[1, 1]` (batch of 1, dimension 1) to an array of shape `[1]`, a 1-element array containing the price
- `np.clip(5.0, 50.0)` : limits the action to be a price within 5 and 50 since the range is theoratically unlimited
    - clip the range after sampling rather than sampling from a bounded distribution
- `next_obs, reward, terminated, truncated, _ = env.step(action_np)` : obtains the state after the nn provides an action

```python
buffer.add(obs = obs,
            action = action.squeeze(0).cpu(),
            log_prob = log_prob.squeeze(0).cpu(),
            reward = reward,
            value = value.squeeze(0).cpu().item(),
            done = float(done))
```
- updating the buffer with all new states, applying `.squeeze(0)` to `action`, `log_prob` and `value` to adjust its shape and converts `done` to a float for computation purposes within buffer

```python
with torch.no_grad():
    last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
    _, _, last_value = model.get_action(last_obs)
    last_value = last_value.squeeze(0).cpu().item()
advantages, returns = buffer.compute_returns(last_value)
obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
```
- This step is to process any partial episodes at the end of the 512 timestep limit.
- e.g. if each episode take 100 steps, the last 12 steps will be an incomplete episode. Therefore, this step will process the last known observation 
    - This processing is done using `with torch.no_grad()` to turn off the computation of gradients so that the model is not updated using incomplete episode data
    - next two lines are then used to obtain the price suggested by the model 
- `advantages, returns = buffer.compute_returns(last_value)` : use the last observation, the incomplete data to calculate the advantages and returns
- `obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)` : converts the data stored in the buffer to learning signals 





In [9]:
model = train(total_timesteps=200_000)

Training on: cpu
Timestep      63 | Episode   20 | Reward:     0.00
Timestep     125 | Episode   40 | Reward:     0.00
Timestep     187 | Episode   60 | Reward:     0.31
Timestep     254 | Episode   80 | Reward:     3.82
Timestep     315 | Episode  100 | Reward:     0.00
Timestep     378 | Episode  120 | Reward:     0.00
Timestep     443 | Episode  140 | Reward:     0.00
Timestep     506 | Episode  160 | Reward:     0.77
Timestep     570 | Episode  180 | Reward:     1.76
Timestep     637 | Episode  200 | Reward:     0.00
Timestep     698 | Episode  220 | Reward:     3.52
Timestep     761 | Episode  240 | Reward:     0.00
Timestep     821 | Episode  260 | Reward:     1.29
Timestep     886 | Episode  280 | Reward:     0.00
Timestep     952 | Episode  300 | Reward:     1.46
Timestep    1018 | Episode  320 | Reward:     3.56
Timestep    1087 | Episode  340 | Reward:     6.18
Timestep    1154 | Episode  360 | Reward:     0.00
Timestep    1221 | Episode  380 | Reward:     0.00
Timestep    12

##### 4.6 Instantiate functions to evaluate model

In [10]:
def evaluate_model(model, n_episodes=50, deterministic=True):
    device = next(model.parameters()).device
    model.eval()
    episode_rewards = []
    ending_inventory = []
    steps_taken = []
    for _ in range(n_episodes):
        env = DynamicPricingEnv()
        obs, _ = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, #convert state to a batched tensor 
                                      device=device).unsqueeze(0) #unsqueeze.(0) inserts a new dimension of size 1 at the 0th position
            with torch.no_grad(): #temporarily disables autograd so that operations inside do not track history or build computation graphs
                mean, std, _ = model.forward(obs_tensor)
                if deterministic:
                    action = mean
                else:
                    action = Normal(mean, std).sample()
            action_np = action.squeeze(0).detach().cpu().numpy()
            action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
            obs, reward, terminated, truncated, _ = env.step(action_np)
            total_reward += reward
            done = terminated or truncated
        episode_rewards.append(total_reward)
        ending_inventory.append(env.inventory)
        steps_taken.append(env.step_count)
    results = {
        "episode_rewards": episode_rewards,
        "mean_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "min_reward": float(np.min(episode_rewards)),
        "max_reward": float(np.max(episode_rewards)),
        "mean_ending_inventory": float(np.mean(ending_inventory)),
        "mean_steps": float(np.mean(steps_taken))}
    print(f"Episodes: {n_episodes}")
    print(f"Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
    print(f"Reward range: [{results['min_reward']:.2f}, {results['max_reward']:.2f}]")
    print(f"Mean ending inventory: {results['mean_ending_inventory']:.2f}")
    print(f"Mean steps per episode: {results['mean_steps']:.2f}")

    return results

In [11]:
eval_results = evaluate_model(model, n_episodes=100, deterministic=True)

Episodes: 100
Mean reward: 39.09 +/- 0.31
Reward range: [36.29, 39.33]
Mean ending inventory: 0.01
Mean steps per episode: 26.78


##### 4.7 Storing the model as a class

In [12]:
class PPOAgent():
    def __init__(self, name: str = "bot", NN=None, env : gym.Env = None): #Store a PyTorch model and device for inference / persistence
        self.name = name
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        if env is not None:
            self.env = env
        else: 
            self.env = DynamicPricingEnv()
        if NN is None:
            self.NN = ActorCritic(self.env.observation_space.shape[0], self.env.action_space.shape[0]).to(self.device)
        else:
            self.NN = NN.to(self.device)
        self.buffer = RolloutBuffer()
        self.optimizer = optim.Adam(self.NN.parameters(), lr=3e-4)

    def train(self, total_timesteps=200_000, n_steps=512):
        self.NN.train() #set to training mode
        obs, _ = self.env.reset()
        episode_reward, episode_count = 0 , 0
        timestep = 0
        while timestep < total_timesteps: # runs until timestep budget is exhausted
            self.buffer.clear() #reset the buffer at the start of every rollout
            for _ in range(n_steps):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    action, log_prob, value = self.NN.get_action(obs_tensor)
                action_np = action.cpu().numpy()[0]
                action_np = np.clip(action_np, 5.0, 50.0)
                next_obs, reward, terminated, truncated, _ = self.env.step(action_np)
                done = terminated or truncated
                self.buffer.add(obs = obs,
                        action = action.squeeze(0).cpu(),
                        log_prob = log_prob.squeeze(0).cpu(),
                        reward = reward,
                        value = value.squeeze(0).cpu().item(),
                        done = float(done))
                episode_reward += reward
                obs = next_obs
                timestep += 1
                if done:
                    episode_count += 1
                    if episode_count % 20 == 0:
                        print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                            f"Reward: {episode_reward:>8.2f}")
                    episode_reward = 0
                    obs, _  = self.env.reset()
            with torch.no_grad():
                last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                _, _, last_value = self.NN.get_action(last_obs)
                last_value = last_value.squeeze(0).cpu().item()
            advantages, returns = self.buffer.compute_returns(last_value)
            obs_t, act_t, lp_t, adv_t, ret_t = self.buffer.to_tensors(advantages, returns, self.device)
            self.ppo_update(obs_t, act_t, lp_t, adv_t, ret_t)
        print("Training complete.")
        return self

    def act(self, obs, deterministic=True):
        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.NN.eval()
        with torch.no_grad():
            mean, std, _ = self.NN.forward(obs_tensor)
            if deterministic:
                action = mean
            else:
                action = Normal(mean, std).sample()
        action_np = action.squeeze(0).cpu().numpy()
        action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
        return action_np

    def play(self, render=False, deterministic=True):
        obs, _ = self.env.reset()
        done = False
        total_reward = 0.0
        total_revenue = 0.0
        steps = 0
        trajectory = []
        while not done:
            action = self.act(obs, deterministic=deterministic)
            price = float(action[0]) if hasattr(action, '__iter__') else float(action) # step FIRST, then read last_demand from env
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            demand = int(self.env.last_demand)# now read the actual demand that generated this reward
            sold = demand  # last_demand already stores units_sold (capped at inventory)
            revenue = price * sold
            trajectory.append((obs, price, demand, float(reward)))
            print(f"Step {steps+1}: Demand: {demand}, Price: {price:.2f}, "
                f"Reward: {reward:.2f}, Sold: {sold}, Revenue: {revenue:.2f}")
            total_reward += reward
            total_revenue += revenue
            steps += 1
            obs = next_obs
            if render:
                self.env.get_latest()
        print(f"Episode finished - Reward: {total_reward:.2f}, Steps: {steps}, "
            f"Ending inventory: {int(self.env.inventory)}")
        print(f"Total revenue generated: ${total_revenue:.2f}")
        return {"total_reward": total_reward, "total_revenue": total_revenue, "steps": steps, 
                "ending_inventory": int(self.env.inventory), "trajectory": trajectory,}

    def evaluate_model(self, n_episodes=50, deterministic=True):
        device = next(self.NN.parameters()).device
        self.NN.eval()
        episode_rewards, ending_inventory = [], []
        steps_taken = []
        for _ in range(n_episodes):
            env = DynamicPricingEnv()
            obs, _ = env.reset()
            done = False
            total_reward = 0.0
            while not done:
                obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    mean, std, _ = self.NN.forward(obs_tensor)
                    if deterministic:
                        action = mean
                    else:
                        action = Normal(mean, std).sample()
                action_np = action.squeeze(0).detach().cpu().numpy()
                action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
                obs, reward, terminated, truncated, _ = env.step(action_np)
                total_reward += reward
                done = terminated or truncated
            episode_rewards.append(total_reward)
            ending_inventory.append(env.inventory)
            steps_taken.append(env.step_count)
        results = {"episode_rewards": episode_rewards, "mean_reward": float(np.mean(episode_rewards)),
            "std_reward": float(np.std(episode_rewards)),"min_reward": float(np.min(episode_rewards)),
            "max_reward": float(np.max(episode_rewards)),"mean_ending_inventory": float(np.mean(ending_inventory)),
            "mean_steps": float(np.mean(steps_taken))}
        print(f"Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
        print(f"Reward range: [{results['min_reward']:.2f}, {results['max_reward']:.2f}]")
        print(f"Mean ending inventory: {results['mean_ending_inventory']:.2f}")
        print(f"Mean steps : {results['mean_steps']:.2f}")
        return results

    #helpers:
    def ppo_update(self, obs, actions, old_log_probs, advantages, returns, 
               clip_range=0.2, ent_coef=0.05, vf_coef=0.5, n_epochs=10, batch_size=64):
        total_steps = obs.shape[0]
        self.NN.train()
        for _ in range(n_epochs):
            indices = torch.randperm(total_steps)
            for start in range(0, total_steps, batch_size):
                idx = indices[start : start + batch_size]
                new_log_probs, values, entropy = self.NN.evaluate(obs[idx], actions[idx])
                ratio = (new_log_probs - old_log_probs[idx]).exp()
                adv = advantages[idx]
                policy_loss = -torch.min(ratio * adv, torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv).mean()
                value_loss = nn.functional.mse_loss(values, returns[idx])
                entropy_loss = -entropy.mean()
                loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.NN.parameters(), max_norm=0.5)
                self.optimizer.step()

In [13]:
agent = PPOAgent(name="bot")
agent.train()


Timestep      63 | Episode   20 | Reward:     0.09
Timestep     129 | Episode   40 | Reward:     0.00
Timestep     194 | Episode   60 | Reward:     2.17
Timestep     257 | Episode   80 | Reward:     1.87
Timestep     320 | Episode  100 | Reward:     0.00
Timestep     381 | Episode  120 | Reward:     0.00
Timestep     445 | Episode  140 | Reward:     0.86
Timestep     508 | Episode  160 | Reward:     1.39
Timestep     572 | Episode  180 | Reward:     0.00
Timestep     634 | Episode  200 | Reward:     0.31
Timestep     700 | Episode  220 | Reward:     0.00
Timestep     765 | Episode  240 | Reward:     0.00
Timestep     827 | Episode  260 | Reward:     2.21
Timestep     893 | Episode  280 | Reward:     0.00
Timestep     957 | Episode  300 | Reward:     3.87
Timestep    1019 | Episode  320 | Reward:     1.75
Timestep    1088 | Episode  340 | Reward:     0.00
Timestep    1159 | Episode  360 | Reward:     1.32
Timestep    1224 | Episode  380 | Reward:     0.00
Timestep    1291 | Episode  400

In [14]:
results = agent.evaluate_model(n_episodes=100)
print(results)

Mean reward: 39.27 +/- 0.29
Reward range: [36.66, 39.53]
Mean ending inventory: 0.01
Mean steps : 25.96
{'episode_rewards': [39.13217197418213, 39.28387676239014, 39.391520500183105, 39.423967399597174, 39.31288955688476, 39.323867454528816, 39.15657077789306, 39.402750625610366, 38.967164535522464, 39.15983711242676, 39.43120155334472, 39.347307662963864, 39.12778572082519, 39.424880447387686, 38.94466289520263, 39.333177146911616, 39.213335876464846, 39.309243927001944, 39.42517723083496, 39.37900283813477, 39.29604698181152, 39.29550701141358, 39.36167537689209, 39.37579730987549, 39.14229331970214, 39.322034149169916, 39.38073383331299, 39.089920120239256, 39.340968437194825, 39.35157421112061, 39.17418933868407, 39.33093147277831, 39.418435821533215, 39.367133445739746, 39.313513031005854, 39.37495796203612, 39.43885639190673, 39.33199401855469, 39.337558860778806, 39.29572841644288, 39.379443740844735, 39.24368724822998, 39.32993247985839, 39.229060668945316, 39.28385425567628, 3

In [15]:
outcome = agent.play()

Step 1: Demand: 5, Price: 42.77, Reward: 1.89, Sold: 5, Revenue: 213.85
Step 2: Demand: 4, Price: 43.18, Reward: 1.53, Sold: 4, Revenue: 172.73
Step 3: Demand: 7, Price: 43.39, Reward: 2.69, Sold: 7, Revenue: 303.75
Step 4: Demand: 6, Price: 43.67, Reward: 2.32, Sold: 6, Revenue: 262.04
Step 5: Demand: 5, Price: 43.75, Reward: 1.94, Sold: 5, Revenue: 218.76
Step 6: Demand: 2, Price: 43.77, Reward: 0.78, Sold: 2, Revenue: 87.55
Step 7: Demand: 5, Price: 43.74, Reward: 1.94, Sold: 5, Revenue: 218.70
Step 8: Demand: 7, Price: 43.79, Reward: 2.72, Sold: 7, Revenue: 306.51
Step 9: Demand: 7, Price: 43.89, Reward: 2.72, Sold: 7, Revenue: 307.25
Step 10: Demand: 1, Price: 44.03, Reward: 0.39, Sold: 1, Revenue: 44.03
Step 11: Demand: 3, Price: 43.98, Reward: 1.17, Sold: 3, Revenue: 131.94
Step 12: Demand: 5, Price: 44.03, Reward: 1.95, Sold: 5, Revenue: 220.14
Step 13: Demand: 6, Price: 44.18, Reward: 2.35, Sold: 6, Revenue: 265.10
Step 14: Demand: 6, Price: 44.47, Reward: 2.37, Sold: 6, Reven

____
### 5. Training the TD3 Model
- TD3 uses 3 networks
    1. Actor Network
        - Outputs a single exact action
        - `state → action`
    2. Critic Network (2 critic networks)
        - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as PPO)
    3. Replay Buffer
        - To store experiences and reuse them, making the TD3 highly sample efficient
        - Allow TD3 to learn from past experiences repeatedly
    4. Target Networks
        - the original network that updates slowly so that the they provide the traget values that the critics learn towards
        
##### TD3 Flow
- `Agent interacts with environment and stores it in replay buffer → TGT Actor predicts the next action → Twin Critic Networks estimates how good the next action is → Lower Q-value generated from the 2 TGT networks is used as training target → Main Critics learns from this value and tries to match this value → Main Actor is trained using the lower of the 2 Critic Values`

Therefore, TGT Networks create Target Values → TGT Values train Main Critic Network → Main Critic trains Main Actor 

#### 5.1 Creating Actor and Critic Networks

In [16]:
class TD3Actor(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()
        )
        
    def forward(self, obs): #needs to be overridden
        return self.net(obs)

In [17]:
class TD3Critic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, obs, action):
        x = torch.cat([obs, action], dim=-1)
        return self.net(x)

##### 5.11 Explanation of code

```python
self.net = nn.Sequential( #actor network
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh())
```

- `nn.sequential()` ensures that the following layers run in sequence
- choice of functions used:
- `nn.Linear(obs_dim, 64)` -> y = Wx + b ; used to expand the 3 value state into 64 values
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.ReLU()` -> y = ReLu(x) = max(0, x)
    - The ReLU function reads every value and limits it to only non-negative number
- `nn.Linear(64, action_dim)` -> y = Wx + b
    - converts the 64 dimensional vector back into a 1 dimensional value, the exact price
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer
- `forward(self, obs)` is the method used to obtain the action based off the observation 

```python
self.net = nn.Sequential( #critic network
            nn.Linear(obs_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1))
```
- `nn.Linear(obs_dim + action_dim, 64)`
    - the neural network needs to read both the observation and the action in order to come up with a measurement for future rewards (critic value)
    - expands it out into 64 values
- `nn.Linear(64, 1)`
    - the 64 values are then compressed back to produce 1 value, the Q-value which is the measurement of future rewards
- `forward(self, obs, action)` is the method used to obtain the Q-value, the estimate of discounted future rewards
    - `x = torch.cat([obs, action], dim = -1)` concatenates the observation and the action into 1 tensor, which is then used as the input for the neural network when `self.net(x)` is called

##### 5.12 Overarching view: PPO vs TD3
- Choice of hidden layers:
    - PPO's action network outputs a distribution instead and then samples it to obtain an action 
    - Therefore, the use of `tanh()` in the hidden layers provides a bound for the distribution
    - However, since the TD3' actor network outputs an action directly, there is no need to limit the bounds with a `tanh()` function in the hidden layers
    - Instead, `ReLU()` can be used for the unboundedness to improve training efficiency
    - If `tanh()` were to be used, when input values are very large or very small, the gradient approaches zero, making the weights in early layers small and barely updated during backprop
- Difference in Critic
    - PPO's critic only estimated and answered "how good is this state", therefore it only read the state
    - TD3's critic estimates and answers "how good is this action in this state", therefore it reads both the state and action 
    - Additionally, TD3 has two critic networks to take the minimum Q-value to prevent overestimation bias

#### 5.2 Creating ReplayBuffer
- used to store up to 100,000 transitions and do not clear out, overriding the oldest entry when full
- since TD3 answers "how good is this action in this state", it does not require episode tracking 

In [18]:
class ReplayBuffer:
    def __init__(self, capacity=100000): 
        self.capacity = capacity
        self.buffer = []
        self.position = 0 #tracks where to write the next transition

    def add(self, obs, action, reward, next_obs, done):
        transition = (obs, action, reward, next_obs,done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
            self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (torch.tensor(np.array(obs), dtype=torch.float32),
                torch.tensor(np.array(actions), dtype=torch.float32),
                torch.tensor(np.array(rewards), dtype=torch.float32).unsqueeze(1),
                torch.tensor(np.array(next_obs), dtype=torch.float32),
                torch.tensor(np.array(dones), dtype=torch.float32).unsqueeze(1))
    def __len__(self):
        return len(self.buffer)

##### 5.21 Explanation of code

```python
def add(self, obs, action, reward, next_obs, done):
    transition = (obs, action, reward, next_obs,done)
    if len(self.buffer) < self.capacity:
        self.buffer.append(transition)
    else:
        self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity
```
- `if len(self.buffer) < self.capacity` : if the buffer still has space
    - `self.buffer.append(transition)` : add the values in and just grow the buffer
- if it is full, rewrites the first output:
    - `self.buffer[self.position] = transition` : overrides the first value as `self.position = 0` until updated
    - `self.position = (self.position + 1) % self.capacity` : advances the write-head by 1

```python
def sample(self, batch_size):
    batch = np.random.sample(self.buffer, batch_size)
    obs, actions, rewards, next_obs, dones = zip(*batch)
    return (torch.tensor(np.array(obs), dtype=torch.float32),
            torch.tensor(np.array(actions), dtype=torch.float32),
            torch.tensor(np.array(rewards), dtype=torch.float32).unsqueeze(1),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(np.array(dones), dtype=torch.float32).unsqueeze(1))
```
- `batch = np.random.sample(self.buffer, batch_size)` : draw a batch_size number of transitions uniformly at random
- `obs, actions, rewards, next_obs, dones = zip(*batch)` : unpacks the list of tuples into 5 seperate tuples, one per field


#### 5.3 Storing agent within a class:

In [19]:
import copy
class TD3Agent:
    def __init__(self, env=None, gamma=0.99, tau=0.005, policy_noise=0.2, noise_clip=0.5,
                 policy_delay=2, expl_noise=0.1, batch_size=256, buffer_capacity=100_000, lr=1e-3):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if env is not None:
            self.env = env
        else:
            self.env = DynamicPricingEnv()
        self.action_low  = float(self.env.action_space.low[0]) #lowest possible value for action 
        self.action_high = float(self.env.action_space.high[0]) #highest possible value for action
        #initialising the 3 main networks
        self.actor   = TD3Actor(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)
        self.critic1 = TD3Critic(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)
        self.critic2 = TD3Critic(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)
        # initialising target networks, frozen copies which are updated slowly via polyak
        self.tgt_actor   = copy.deepcopy(self.actor)
        self.tgt_critic1 = copy.deepcopy(self.critic1)
        self.tgt_critic2 = copy.deepcopy(self.critic2)
        # target networks never receive gradient updates directly
        for net in [self.tgt_actor, self.tgt_critic1, self.tgt_critic2]:
            for param in net.parameters():
                param.requires_grad = False
        self.actor_opt   = optim.Adam(self.actor.parameters(), lr=lr) #optimisers
        self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity) #buffer
        # hyperparameters
        self.gamma        = gamma
        self.tau          = tau
        self.policy_noise = policy_noise
        self.noise_clip   = noise_clip
        self.policy_delay = policy_delay
        self.expl_noise   = expl_noise
        self.batch_size   = batch_size
        self.total_updates = 0  # tracks how many critic updates done, for policy_delay

    def act(self, obs, add_noise=True): #method to produce an action given an observation 
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.actor.eval()
        with torch.no_grad():
            raw = self.actor(obs_t)# tanh output in (-1, 1)
        self.actor.train()
        price = self.rescale(raw.cpu().numpy()[0])
        if add_noise:
            noise = np.random.normal(0, self.expl_noise *
                                     (self.action_high - self.action_low),
                                     size=price.shape)
            price = np.clip(price + noise, self.action_low, self.action_high)
        return price.astype(np.float32)

    def play(self, render=False, deterministic=True):
        obs, _ = self.env.reset()
        done = False
        total_reward = 0.0
        total_revenue = 0.0
        steps = 0
        trajectory = []
        while not done:
            action = self.act(obs, add_noise=not deterministic)
            price = float(action[0]) if hasattr(action, '__iter__') else float(action)
            demand = self.env.demand(price)
            sold = min(demand, int(self.env.inventory))
            revenue = price * sold
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            trajectory.append((obs, price, int(demand), float(reward)))
            print(f"Demand: {demand}, Price: {price:.2f}, Reward: {reward:.2f}, Sold: {sold}, Revenue: {revenue:.2f}")
            total_reward += reward
            total_revenue += revenue
            steps += 1
            obs = next_obs
            if render:
                self.env.get_latest()
        print(f"Episode finished - Reward: {total_reward:.2f}, Steps: {steps}, Ending inventory: {int(self.env.inventory)}")
        print(f"Total revenue generated: ${total_revenue:.2f}")
        return {
            "total_reward": total_reward,
            "total_revenue": total_revenue,
            "steps": steps,
            "ending_inventory": int(self.env.inventory),
            "trajectory": trajectory,
        }

    def td3_update(self): #function to update the model during training 
        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size) #unpacks a random batch selected from the replay buffer
        obs      = obs.to(self.device) # move everything to the correct device
        actions  = actions.to(self.device)
        rewards  = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones    = dones.to(self.device)
        with torch.no_grad(): #to update critic
            raw_next = self.tgt_actor(next_obs) # target actor suggests the next action from next_obs within range (-1, 1)
            next_actions = self.action_low + (raw_next + 1.0) * 0.5 * (self.action_high - self.action_low)
            noise = torch.randn_like(next_actions) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_actions = (next_actions + noise).clamp(self.action_low, self.action_high)
            q1_next = self.tgt_critic1(next_obs, next_actions) # twin critics evaluate (next_obs, next_actions)
            q2_next = self.tgt_critic2(next_obs, next_actions)
            q_next = torch.min(q1_next, q2_next) #use min as the target
            target_q = rewards + self.gamma * (1.0 - dones) * q_next # bellman target: if episode ended (done=1), no future reward
        q1_current = self.critic1(obs, actions) # compute current Q estimates and MSE against the target
        q2_current = self.critic2(obs, actions)
        critic1_loss = nn.functional.mse_loss(q1_current, target_q)
        critic2_loss = nn.functional.mse_loss(q2_current, target_q)
        self.critic1_opt.zero_grad() # backpropagate critic 1
        critic1_loss.backward()
        self.critic1_opt.step()
        self.critic2_opt.zero_grad() # backpropagate critic 2
        critic2_loss.backward()
        self.critic2_opt.step()
        self.total_updates += 1
        if self.total_updates % self.policy_delay == 0: #update the policy
            raw_actions = self.actor(obs)
            actor_actions = self.action_low + (raw_actions + 1.0) * 0.5 * (self.action_high - self.action_low)
            actor_loss = -self.critic1(obs, actor_actions).mean() # gradient ascent on Q → gradient descent on negative Q
            self.critic1_opt.zero_grad()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()  # only actor weights are updated, not critic1
            for main, target in [(self.actor,   self.tgt_actor), (self.critic1, self.tgt_critic1), (self.critic2, self.tgt_critic2)]:#polyak updates, slowly updating the TGT network
                for p_main, p_tgt in zip(main.parameters(), target.parameters()):
                    p_tgt.data.mul_(1.0 - self.tau)
                    p_tgt.data.add_(self.tau * p_main.data)
    
    def train(self, total_timesteps=200_000, learning_starts=1_000, log_every=20): #function to train the model 
        obs, _ = self.env.reset()
        episode_reward = 0.0
        episode_count  = 0
        for timestep in range(1, total_timesteps + 1):
            # before learning_starts, take random actions to pre-fill buffer
            if timestep < learning_starts:
                action = self.env.action_space.sample()
            else:
                action = self.act(obs, add_noise=True)
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            # store transition — use terminated (not done) for the done flag
            # so that a truncated episode doesn't incorrectly zero out future rewards
            self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))
            episode_reward += reward
            obs = next_obs
            if done:
                episode_count += 1
                if episode_count % log_every == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0.0
                obs, _ = self.env.reset()
            # only start learning once the buffer has enough samples
            if timestep >= learning_starts:
                self.td3_update()
        print("Training complete.")
        return self

    def evaluate(self, n_episodes=100): #function to evaluate the model
        rewards, inventories, steps = [], [], []

        for _ in range(n_episodes):
            env = DynamicPricingEnv()
            obs, _ = env.reset()
            done = False
            total_reward = 0.0
            while not done:
                action = self.act(obs, add_noise=False) # deterministic at eval
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                done = terminated or truncated
            rewards.append(total_reward)
            inventories.append(env.inventory)
            steps.append(env.step_count)
        print(f"Episodes : {n_episodes}")
        print(f"Mean reward : {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")
        print(f"Reward range: [{np.min(rewards):.2f}, {np.max(rewards):.2f}]")
        print(f"Mean ending inventory: {np.mean(inventories):.2f}")
        print(f"Mean steps : {np.mean(steps):.2f}")

    def rescale(self, raw_action): #helper function to rescale price: convert value from (-1, 1) to (5, 50)
        return self.action_low + (raw_action + 1.0) * 0.5 * (self.action_high - self.action_low)

##### 5.31 Explanation of code

- Hyperparameters:
    1. `gamma=0.99` :  how much the agent cares about future rewards vs immediate reward
        - future rewards are worth 0.99 of immediate ones, almost equal importance
    2. `tau=0.005` : the polyak averaging rate, how fast the tgt networks drift towards the main one
        - `weights of TGT = tau * weights of main + (1 - tau) * weights of TGT`
        - if tau is too high: TGT changes quickly, causing less stability, close to directly copying the main network
        - if tau too low: TGT changes slowly, more stability but learns too slowly
    3. `policy_noise=0.2`: the std dev of noise added to the target actor's actions when computing training target
        - noise is needed to create variation, forcing the critic TGT to be an average over nearby actions, smoothing out any spikes in Q-values
    4. `noise_clip=0.5` : limits how far the target policy smoothing noise can push the action
        - `noise_clip = 0.5` makes it such that the noise is bounded to ±0.5 around the actor's chosen action
    5. `policy_delay=2` : controls how many critic updates happen per actor update
        - `policy_delay=2` means that the actor updatess every 2nd critic step
        - if policy_update too low (e.g. 1), the actor updates every step
        - if policy_update too high (e.g. 5), the actor updates less frequently 
        - delaying is needed so that the actor is trained to maximise the critic's Q-value. If the critic is inaccurate in the early stages, error is compounded if there is no delay
    6. `expl_noise=0.1` : exploration noise during data collection, the std dev of noise added to the actor's output
        - `noise_std = expl_noise * range = 0.1 * (5-0.5) = 4.5` : this means that the prices vary by roughly ±4.5 around what the actor network suggests
        - used purely for exploration during training and turned off at evaluation
    7. `batch_size=256` : minibatches size for updates
        - controls how many transitions are sampled from the replay buffer per update step
        - `batch_size = 256` means that 256 transitions are sampled and used for one update
    8. `buffer_capacity=100_000` : controls the size of the buffer, how many past transitions the agent remembers
        - if too large, buffer contains old experiences when the policy was poor and random, confuse learning with outdated experiences
        - if too small, the buffer overfits to recent experiences, losing the diversity of the earlier experiences
    9. `lr=1e-3` : the learning rate for the optimizer, controls how large each gradient step is for all three optimizers

```python
def act(self, obs, add_noise=True): 
    obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
    self.actor.eval()
    with torch.no_grad():
        raw = self.actor(obs_t) # tanh output in range (-1, 1)
    self.actor.train()
    price = self.rescale(raw.cpu().numpy()[0])   # shape (1,), value in (5, 50)
    if add_noise:
        noise = np.random.normal(0, self.expl_noise *
                                     (self.action_high - self.action_low),
                                     size=price.shape)
        price = np.clip(price + noise, self.action_low, self.action_high)
    return price.astype(np.float32)
```
- the method `act(self, obs, add_noise = True)` takes in an action and returns a price
    - `add_noise = True` by default for training and set to false during evaluation to make it deterministic during evaluation
- `obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)` : converts the numpy array that represents the observation into a PyTorch tensor
    - `device = self.device` moves it to the correct device
    - `.unsqueeze(0)` adds a batch dimension, changing the shape from `[3]` to `[1,3]` since the model expects the shape to be `[batch, feature]`
- `self.actor.eval()`: switches to evaluation mode, disablinh dropout/batchnorm if present
- `with torch.no_grad()` turns off computation and `raw = self.actor(obs_t)` obtains a tensor shaped `[1,1]` with a value in the range (-1, 1) from the final `Tanh()` function
    - computation of gradient is turned off as we only need the output value and not the gradients
- `self.actor.train()` : switches back to training mode
- `price = self.rescale(raw.cpu().numpy()[0])` : converts the price from the output of the actor network
    - `raw.cpu()` moves the tensor to the CPU so that numpy can access it, `.numpy()` converts the tensor back to a numpy array and `[0]` indexed it to get the output from the last `tanh()` function
    - `.rescale()` converts the value from the `tanh()` function into a price
- `if add_noise` runs the inner block to add noise to the price (makes the function non-deterministic)
    - `np.random.normal(0, self.expl_noise * (self.action_high - self.action_low), size=price.shape)` : generates gaussian noise with `std = 0.1 * 45 = 4.5` and adds it to the price
    - `np.clip(price + noise, self.action_low, self.action_high)` : clips the range of the price to be within the range of `(5, 50)`, done incase noise is too high or action is too high and exceeds the price ceiling

```python
def td3_update(self): #function to update the model during training 
    obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size)
    obs = obs.to(self.device) # move everything to the correct device
    actions = actions.to(self.device)
    rewards = rewards.to(self.device)
    next_obs = next_obs.to(self.device)
    dones = dones.to(self.device)
    with torch.no_grad(): #to update critic
        raw_next = self.tgt_actor(next_obs) # (-1, 1)
        next_actions = self.action_low + (raw_next + 1.0) * 0.5 * (self.action_high - self.action_low)
        noise = torch.randn_like(next_actions) * self.policy_noise
        noise = noise.clamp(-self.noise_clip, self.noise_clip)
        next_actions = (next_actions + noise).clamp(self.action_low, self.action_high)
        q1_next = self.tgt_critic1(next_obs, next_actions)
        q2_next = self.tgt_critic2(next_obs, next_actions)
        q_next = torch.min(q1_next, q2_next) #lower q value used
        target_q = rewards + self.gamma * (1.0 - dones) * q_next #q value that both critic mains are trained to work towards
```

- `obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size)` : samples from the buffer for a number of samples equal to `self.batch_size`
    - unpacks them into respective tensors
- `.to(self.device)` : to move the respective tensors into the devide that the network is on
- `with torch.no_grad()` : performs the actions in the block without building a computation graph
    - the block is meant to update the critic so the target is a fixed number (no need to compute gradients)
- `raw_next = self.tgt_actor(next_obs)` obtains the action from the TGT Actor, value in the range (-1, 1)
- `next_actions = self.action_low + (raw_next + 1.0) * 0.5 * (self.action_high - self.action_low)` : rescales the value back to in the range (5, 50)
- The functions in the next 3 lines are meant to smoothen out the policy
    - `noise = torch.randn_like(next_actions) * self.policy_noise` : creates a tensor in the same shape as `next_actions`, filling it with random Gaussian values   
        - `* self.policy_noise` scales all the values in the tensor by `policy_noise` 
    - `noise = noise.clamp(-self.noise_clip, self.noise_clip)` : clamps the noise to the predetermined range to prevent extreme values which can make training unstable 
    - `next_actions = (next_actions + noise).clamp(self.action_low, self.action_high)` : adds the values inside the `noise` tensor to the `next_actions` tensor, value-wise addition. `.clamp()` then limits the range back 
    - by adding noise, instead of evaluating exact next price, the critic will target averages over a small neighbourhood of nearby prices, smoothing out artificially sharp Q-value peaks
- `q1_next = self.tgt_critic1(next_obs, next_actions)` and `q2_next = self.tgt_critic2(next_obs, next_actions)` : using the TGT critic networks to evaluate the actions of the TGT actor network
- `q_next  = torch.min(q1_next, q2_next)` : obtains the lower value, the main point of a TD3, where the lower value is used to prevent overestimation bias
- `target_q = rewards + self.gamma * (1.0 - dones) * q_next` : The bellman equation
    - for each transition, `targeted Q = immediate reward + future reward` 
    - `(1.0-dones)` is the terminal switch, since `dones = 1` when the episode ends, causing the equation to be `target_q = rewards`
    - this q-value is the value that both main critics are trained to predict 

```python
q1_current = self.critic1(obs, actions) #after exiting the with torch.no_grad() block
q2_current = self.critic2(obs, actions)
critic1_loss = nn.functional.mse_loss(q1_current, target_q)
critic2_loss = nn.functional.mse_loss(q2_current, target_q)
self.critic1_opt.zero_grad()
critic1_loss.backward()
self.critic1_opt.step()
self.critic2_opt.zero_grad()
critic2_loss.backward()
self.critic2_opt.step()
self.total_updates += 1
```
- `q1_current = self.critic1(obs, actions)` and `q2_current = self.critic2(obs, actions)` : compute the current q-value estimates
    - using the current critic networks, calculate the q-values so that it can be compared with the values calculated by the TGT critic networks
- `critic1_loss = nn.functional.mse_loss(q1_current, target_q)` and `critic2_loss = nn.functional.mse_loss(q2_current, target_q)` : calculates the mean squared error between the main critic's current prediction and the `bellman target`
    - the MSE is where the `target critic` learns from the `main critic`
- `self.critic1_opt.zero_grad()` : clears accumulated gradients from previous steps
- `critic1_loss.backward()` : computes new gradients by differentiating the loss throught the network
    - applying backpropogation to minimise the `critic1_loss` equation
    - this is the key process that helps the main critic learn from the target critic
- `self.critic1_opt.step()` : applies the weight update to the network

```python
if self.total_updates % self.policy_delay == 0:
    raw_actions = self.actor(obs)
    actor_actions = self.action_low + (raw_actions + 1.0) * 0.5 * (self.action_high - self.action_low)
    actor_loss = -self.critic1(obs, actor_actions).mean()
    self.actor_opt.zero_grad()
    actor_loss.backward()
    self.actor_opt.step()
    for main, target in [(self.actor,   self.tgt_actor), 
                        (self.critic1, self.tgt_critic1),
                        (self.critic2, self.tgt_critic2)]:
        for p_main, p_tgt in zip(main.parameters(), target.parameters()):
            p_tgt.data.mul_(1.0 - self.tau)
            p_tgt.data.add_(self.tau * p_main.data)
```
- `if self.total_updates % self.policy_delay == 0` : only enters this block to update the policy according to the policy_delay indicator
- `raw_actions = self.actor(obs)` : obtains the action from the main actor network in the range (-1, 1)
- `actor_actions = self.action_low + (raw_actions + 1.0) * 0.5 * (self.action_high - self.action_low)` : scales the action back to the range of the (5, 50)
- `actor_loss = -self.critic1(obs, actor_actions).mean()` : evaluates the actor's action based off the critic
    - `.mean()` obtains the mean of the Q-values across the whole batch  
    - multiplies by -1 to flip the sign of the equation : needed since gradient descent only applies for minimising an equation, this allows the `actor_loss` equation to be minimised
- `self.actor_opt.zero_grad()` : clears previously accumulated gradients
- `actor_loss.backward()` : back propogate the `actor_loss` equation 
    - gradients flow through critic1 into actor_actions andd then back into the actor's weights
- `self.actor_opt.step()` : updates the actor network's weights
    * critic1's own weights are NOT updated herem only the actor's weights since the optimizer used here is `self.actor_opt`
- `for main, target in [(self.actor, self.tgt_actor), (self.critic1, self.tgt_critic1), (self.critic2, self.tgt_critic2)]` : iterates over all three network pairs (actor, critic1, critic2) to update the TGT networks
    - this block to update TGT networks is inside the block for updating policy to ensure that all three TGT networks are in sync with each other
- `p_tgt.data.mul_(1.0 - self.tau)` : scales the target network's weight down
- `p_tgt.data.add_(self.tau * p_main.data)` : adds the main network's weight to the TGT network's weight

```python
def train(self, total_timesteps=200_000, learning_starts=1_000, log_every=20):
    obs, _ = self.env.reset()
    episode_reward = 0.0
    episode_count  = 0
    for timestep in range(1, total_timesteps + 1):
            # before learning_starts, take random actions to pre-fill buffer
        if timestep < learning_starts:
            action = self.env.action_space.sample()
        else:
            action = self.act(obs, add_noise=True)
        next_obs, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated
        self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))
        episode_reward += reward
        obs = next_obs
        if done:
            episode_count += 1
            if episode_count % log_every == 0:
                print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                        f"Reward: {episode_reward:>8.2f}")
            episode_reward = 0.0
            obs, _ = self.env.reset()
        if timestep >= learning_starts:
            self.td3_update()
    print("Training complete.")
    return self
```

- Parameters required
    - `total_timesteps=200_000` : number of times the loop within will run for, uses one outer iterative loop only 
    - `learning_starts=1_000` : minimum number of timesteps before model starts learning
    - `log_every=20` : prints out every 20 episodes
- `for timestep in range(1, total_timesteps + 1)` : starts iteration from 1 so that the `timestep < learning_starts` checker will correctly skip the first 1000 steps and `timestep % log_every` produces a clean episode count
- `if timestep < learning_starts:` , `action = self.env.action_space.sample()` : at the early stages of training, the model takes a completely random sampled action until it crosses the `learning_starts` threshold
    - this ensures that the replay buffer starts with diverse experiences rather than the actor's initial (random) policy output which tends to be concentrated near the middle
- `action = self.act(obs, add_noise=True)` : once it crosses the treshold, the model utilises the network's actor network to obtain an action based off the observed state
    - `add_noise = True` to smoothen the policy and minimise the impacts of spikes in Q values
- `next_obs, reward, terminated, truncated, _ = self.env.step(action)` : obtains the next state of the environment based off the action  
- `done = terminated or truncated` : combines the 2 boolean flag into 1
- `self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))` : adds the experience into the replay buffer

- `if timestep >= learning_starts:`, `self.td3_update()` : once the threshold is crossed, learning starts and `td3_update()` is called to train the model during each experience

```python
def evaluate(self, n_episodes=100): #function to evaluate the model
    rewards, inventories, steps = [], [], []
    for _ in range(n_episodes):
        obs, _ = self.env.reset()
        done = False
        total_reward = 0.0
        while not done:
            action = self.act(obs, add_noise=False)  
            obs, reward, terminated, truncated, _ = self.env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
        inventories.append(env.inventory)
        steps.append(env.step_count)
    print(f"Episodes : {n_episodes}")
    print(f"Mean reward : {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")
    print(f"Reward range: [{np.min(rewards):.2f}, {np.max(rewards):.2f}]")
    print(f"Mean ending inventory: {np.mean(inventories):.2f}")
    print(f"Mean steps : {np.mean(steps):.2f}")
```

- `rewards, inventories, steps = [], [], []` : instantiating lists to store data
- `for _ in range(n_episodes)` : run `n_episodes` number of episodes
- `while not done` : keeps obtaining actions from the network instead of resetting the environment until the episode is done
    - `action = self.act(obs, add_noise=False)` : obtain the action, `add_noise = False` so that a deterministic action is obtained


##### 5.4 Sample run of instantiating and training the model

In [21]:
TD3agent = TD3Agent()
TD3agent.train(); 

Timestep     125 | Episode   20 | Reward:    11.76
Timestep     246 | Episode   40 | Reward:    13.46
Timestep     372 | Episode   60 | Reward:    18.43
Timestep     492 | Episode   80 | Reward:    15.39
Timestep     627 | Episode  100 | Reward:    19.06
Timestep     751 | Episode  120 | Reward:    14.56
Timestep     886 | Episode  140 | Reward:     9.58
Timestep    1011 | Episode  160 | Reward:    25.69
Timestep    1342 | Episode  180 | Reward:    17.00
Timestep    1461 | Episode  200 | Reward:    22.22
Timestep    1590 | Episode  220 | Reward:    21.60
Timestep    1714 | Episode  240 | Reward:    23.26
Timestep    1849 | Episode  260 | Reward:    19.87
Timestep    1982 | Episode  280 | Reward:    26.65
Timestep    2113 | Episode  300 | Reward:    21.14
Timestep    2246 | Episode  320 | Reward:    25.08
Timestep    2389 | Episode  340 | Reward:    27.01
Timestep    2543 | Episode  360 | Reward:    24.95
Timestep    2705 | Episode  380 | Reward:    24.53
Timestep    2887 | Episode  400

In [ ]:
TD3agent.evaluate()

Episodes : 100
Mean reward : 38.93 +/- 0.10
Reward range: [38.35, 39.16]
Mean ending inventory: 66.00
Mean steps : 1.00


In [ ]:
outcome = TD3agent.play()

Demand: 5, Price: 42.66, Reward: 1.51, Sold: 5, Revenue: 213.31
Demand: 6, Price: 42.75, Reward: 2.27, Sold: 6, Revenue: 256.51
Demand: 8, Price: 43.26, Reward: 3.83, Sold: 8, Revenue: 346.05
Demand: 6, Price: 43.66, Reward: 0.77, Sold: 6, Revenue: 261.98
Demand: 4, Price: 43.93, Reward: 3.50, Sold: 4, Revenue: 175.72
Demand: 6, Price: 42.87, Reward: 1.14, Sold: 6, Revenue: 257.24
Demand: 3, Price: 42.13, Reward: 1.86, Sold: 3, Revenue: 126.40
Demand: 5, Price: 41.91, Reward: 2.21, Sold: 5, Revenue: 209.55
Demand: 7, Price: 41.56, Reward: 2.92, Sold: 7, Revenue: 290.92
Demand: 8, Price: 43.04, Reward: 0.76, Sold: 8, Revenue: 344.33
Demand: 3, Price: 43.92, Reward: 1.17, Sold: 3, Revenue: 131.76
Demand: 4, Price: 44.91, Reward: 1.60, Sold: 4, Revenue: 179.65
Demand: 5, Price: 44.92, Reward: 2.00, Sold: 5, Revenue: 224.58
Demand: 3, Price: 44.92, Reward: 1.20, Sold: 3, Revenue: 134.75
Demand: 1, Price: 44.92, Reward: 1.60, Sold: 1, Revenue: 44.92
Demand: 3, Price: 44.92, Reward: 1.20, So